# Наивный Байес — практическое задание

В этом ноутбуке рассматривается метод **наивного байесовского классификатора** на примере реального набора данных с Kaggle.
Ноутбук рассчитан на выполнение в **Google Colab**.


## Теоретическое введение

### 1. Теорема Байеса

Основой метода служит **теорема Байеса**:
$$
P(y \mid x)=\frac{P(x \mid y)\,P(y)}{P(x)}.
$$

Здесь:
- $P(y \mid x)$ — апостериорная вероятность класса $y$ после наблюдения объекта $x$;
- $P(x \mid y)$ — правдоподобие, то есть вероятность наблюдать признаки $x$ при условии, что объект принадлежит классу $y$;
- $P(y)$ — априорная вероятность класса;
- $P(x)$ — нормирующий множитель.

В задаче классификации нас обычно интересует не точное значение $P(x)$, а то, какой класс даёт наибольшее значение апостериорной вероятности.

### 2. Идея наивного Байеса

Пусть объект описывается признаками
$$
x=(x_1,x_2,\dots,x_d).
$$

Тогда по теореме Байеса:
$$
P(y \mid x_1,\dots,x_d)\propto P(y)\,P(x_1,\dots,x_d \mid y).
$$

Главное допущение метода состоит в том, что признаки **условно независимы при фиксированном классе**:
$$
P(x_1,\dots,x_d \mid y)=\prod_{j=1}^{d} P(x_j \mid y).
$$

Это предположение редко выполняется строго, поэтому классификатор называется **наивным**. Тем не менее, на практике он часто работает удивительно хорошо, особенно в задачах текстовой классификации.

### 3. Правило классификации

С учётом предположения независимости получаем:
$$
P(y \mid x)\propto P(y)\prod_{j=1}^{d} P(x_j \mid y).
$$

Тогда предсказание выполняется по правилу максимума:
$$
\hat y=\arg\max_{y} \left( P(y)\prod_{j=1}^{d} P(x_j \mid y) \right).
$$

На практике для численной устойчивости часто переходят к логарифмам:
$$
\log P(y \mid x)=\log P(y)+\sum_{j=1}^{d}\log P(x_j \mid y)+C,
$$
где $C$ не зависит от класса и потому не влияет на выбор $\hat y$.

### 4. Виды наивного Байеса

В библиотеке `scikit-learn` чаще всего используются три варианта:

#### GaussianNB
Применяется к непрерывным признакам и предполагает, что при каждом классе признаки имеют нормальное распределение:
$$
x_j \mid y \sim \mathcal{N}(\mu_{jy}, \sigma_{jy}^2).
$$

#### MultinomialNB
Подходит для счётчиков и частот, например:
- числа вхождений слов в тексте;
- bag-of-words представления документов.

Именно этот вариант особенно популярен в задачах классификации текстов.

#### BernoulliNB
Используется для бинарных признаков, например:
- есть ли слово в документе;
- содержит ли сообщение ссылку;
- присутствует ли определённый шаблон.

### 5. Почему метод хорошо подходит для текста

После векторизации текста документ представляется набором признаков:
- либо числами вхождений слов;
- либо бинарными индикаторами присутствия слов;
- либо TF-IDF-весами.

Наивный Байес хорошо работает в такой постановке, потому что:
- признаки часто очень разреженные;
- размерность пространства высокая;
- обучение модели выполняется очень быстро;
- модель даёт сильный базовый результат даже без сложной настройки.

### 6. Сглаживание Лапласа

Если некоторое слово не встречалось в обучающей выборке для конкретного класса, то соответствующая вероятность может стать нулевой.
Это приводит к обнулению всего произведения вероятностей.

Чтобы избежать этого, используют **сглаживание Лапласа**:
$$
P(x_j \mid y)=\frac{N_{jy}+\alpha}{N_y+\alpha m},
$$
где:
- $N_{jy}$ — число наблюдений признака в классе $y$;
- $N_y$ — суммарное число наблюдений в классе;
- $m$ — число возможных значений;
- $\alpha>0$ — параметр сглаживания.

В `MultinomialNB` этот параметр задаётся как `alpha`.

### 7. Оценка качества классификации

Для оценки качества модели обычно используют:
- **accuracy** — долю правильных предсказаний;
- **confusion matrix** — матрицу ошибок;
- **precision** — точность по положительному классу;
- **recall** — полноту;
- **f1-score** — гармоническое среднее между precision и recall.

В задаче фильтрации спама особенно важно смотреть не только на accuracy, но и на precision/recall по классу `spam`.

### 8. Преимущества и ограничения метода

Преимущества:
- простота;
- высокая скорость обучения и предсказания;
- хорошая работа на текстовых данных;
- устойчивый базовый результат.

Ограничения:
- предположение условной независимости признаков часто нарушается;
- качество может снижаться, если важны сложные зависимости между признаками;
- вероятности модели не всегда хорошо калиброваны.

### 9. Почему этот метод важен

Наивный Байес полезен для изучения, потому что:
- напрямую связан с теорией вероятностей;
- показывает, как байесовские идеи применяются в машинном обучении;
- является одним из классических методов текстовой классификации;
- служит сильной и очень быстрой базовой моделью.


## Набор данных

Для работы используется набор данных **SMS Spam Collection Dataset**:
[https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

По описанию набора данных, он содержит **5574 SMS-сообщения**, размеченные как `ham` или `spam`, а в исходном CSV обычно используются столбцы `v1` и `v2`, где `v1` — метка класса, а `v2` — текст сообщения.

Целевая переменная:
- `label` — спам или не спам.

Признаки:
- `message` — текст SMS-сообщения.

Этот датасет хорошо подходит для изучения наивного Байеса, потому что:
- задача является задачей бинарной классификации;
- данные текстовые;
- `MultinomialNB` является классическим методом для bag-of-words представления

Самостоятельно выберите способ загрузить данные и сделать их доступными в блокноте с заданием.

Один из вариантов - скачивание через Kaggle API, используя свой токен. Ещё один вариант - скачать файл вручную, выложить его на Google-диск и потом подмонтировать Google-диск в среду Colab. Ниже приведены инструкции.

## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную настройку Kaggle и Colab.
1. Откройте Kaggle, зайдите в свой профиль и создайте API-токен.
2. В Colab откройте вкладку **Secrets**.
3. Создайте два секрета:
- `KAGGLE_USERNAME`
- `KAGGLE_KEY`

Не забудьте разрешить доступ к этим секретам из блокнота.

Далее следуйте инструкциям в коде ниже.


In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
!kaggle datasets list -s "sms spam collection" | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml"
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

DATASET = "uciml/sms-spam-collection-dataset"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
print("Содержимое папки после скачивания:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)


## Альтернативный вариант — скачивание файла вручную

- скачайте CSV-файл с Kaggle вручную;
- поместите его в папку `sci-prog-and-ml` на Google Диске;
- подмонтируйте Google Диск в Colab и перейдите в эту папку.

После этого файл можно открывать через `pandas.read_csv(...)`.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')

print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)


## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Найдите CSV-файл с SMS-сообщениями и загрузите его в `pandas.DataFrame`.
2. Определите:
   - число объектов;
   - список столбцов;
   - наличие пропущенных значений;
   - распределение классов.
3. Приведите таблицу к удобному виду:
   - оставьте столбцы с меткой класса и текстом;
   - переименуйте их в `label` и `message`.

### Часть 2. Подготовка данных
1. Преобразуйте метки классов:
   - `ham` → 0,
   - `spam` → 1.
2. Разделите данные на обучающую и тестовую выборки.
3. Выполните векторизацию текста:
   - сначала через `CountVectorizer`;
   - затем, по желанию, сравните с `TfidfVectorizer`.

### Часть 3. Построение модели
1. Обучите модель `MultinomialNB`.
2. Получите предсказания на тестовой выборке.
3. Попробуйте изменить параметр сглаживания `alpha`.

### Часть 4. Оценка качества
1. Вычислите accuracy.
2. Постройте confusion matrix.
3. Выведите classification report.
4. Проанализируйте качество распознавания класса `spam`.

### Часть 5. Исследование параметров
1. Сравните результаты для нескольких значений `alpha`, например:
   - 0.1,
   - 0.5,
   - 1.0,
   - 2.0.
2. Сравните:
   - `CountVectorizer + MultinomialNB`;
   - `TfidfVectorizer + MultinomialNB`.
3. Сделайте вывод, какой вариант оказался лучше на данном наборе данных.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import numpy as np
import pandas as pd
```

### Как найти нужный CSV-файл

В датасете может быть несколько файлов. Удобно выбрать CSV, в котором есть столбцы `v1` и `v2`:

```python
csv_files = glob.glob("*.csv")

for path in csv_files:
    tmp = pd.read_csv(path, encoding="latin-1")
    if {"v1", "v2"}.issubset(tmp.columns):
        df = tmp
        break
```

### Подготовка данных
```python
df = df[["v1", "v2"]].copy()
df.columns = ["label", "message"]
df["label"] = df["label"].map({"ham": 0, "spam": 1})
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
```

### Векторизация текста
Для преобразования текста в числовые признаки используйте:
```python
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
```

### Модель
```python
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB(alpha=1.0)
model.fit(X_train_vec, y_train)
```

### Метрики качества
```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
```

### Удобный способ работы
Полезно построить `Pipeline`, который объединяет:
- векторизацию текста;
- модель наивного Байеса.

Тогда можно легко сравнивать разные варианты.


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. В чём состоит предположение условной независимости признаков в наивном Байесе?
2. Почему метод `MultinomialNB` хорошо подходит для текстов после bag-of-words векторизации?
3. Зачем нужен параметр сглаживания `alpha`?
4. Почему в задаче фильтрации спама недостаточно смотреть только на accuracy?
5. Какие типы ошибок для антиспам-фильтра особенно неприятны на практике?


## Дополнительные задания

### Задание 1
Сравните `CountVectorizer` и `TfidfVectorizer` при одинаковом `alpha`.

### Задание 2
Подберите параметр `alpha` на более плотной сетке значений.

### Задание 3
Ограничьте словарь:
- `max_features=1000`,
- `max_features=3000`,
- `max_features=5000`,
и сравните качество.

### Задание 4
Попробуйте удалить стоп-слова и исследуйте, как это повлияет на качество модели.

### Задание 5
Посмотрите на сообщения, которые модель ошибочно считает спамом, и попробуйте объяснить, почему это происходит.
